# 06 Embeddings and Vector Store

## Σκοπός
Σε αυτό το notebook:

- φορτώνονται τα chunks που παρήχθησαν στο προηγούμενο βήμα
- δημιουργούνται embeddings για κάθε chunk
- αποθηκεύονται embeddings και metadata
- δημιουργείται vector store με FAISS
- αποθηκεύεται το index για χρήση στο retrieval

Το notebook είναι σχεδιασμένο ώστε να λειτουργεί τόσο σε pilot run όσο και σε full corpus.

In [ ]:
# Uncomment if needed
# !pip install -q sentence-transformers faiss-cpu pyarrow tqdm hf_xet

In [ ]:
from pathlib import Path
import json
import warnings
import pickle
import re

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import faiss
import torch
from sentence_transformers import SentenceTransformer

In [ ]:
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 140)

IN_KAGGLE = Path("/kaggle/working").exists()
print("IN_KAGGLE:", IN_KAGGLE)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
CHUNKS_DIR = PROCESSED_DIR / "chunks"
EMBEDDINGS_DIR = PROCESSED_DIR / "embeddings"
OUTPUTS_DIR = BASE_DIR / "outputs"

CHUNKS_CSV_PATH = CHUNKS_DIR / "financebench_chunks.csv"
CHUNKS_PARQUET_PATH = CHUNKS_DIR / "financebench_chunks.parquet"
CHUNKING_MANIFEST_PATH = CHUNKS_DIR / "chunking_manifest.csv"

EMBEDDINGS_METADATA_CSV_PATH = EMBEDDINGS_DIR / "chunk_embeddings_metadata.csv"
EMBEDDINGS_MATRIX_NPY_PATH = EMBEDDINGS_DIR / "chunk_embeddings.npy"
FAISS_INDEX_PATH = EMBEDDINGS_DIR / "financebench_faiss.index"
EMBEDDINGS_MANIFEST_PATH = EMBEDDINGS_DIR / "embeddings_manifest.csv"
EMBEDDINGS_STATS_PATH = EMBEDDINGS_DIR / "embeddings_stats.json"
ROW_MAPPING_PATH = EMBEDDINGS_DIR / "faiss_row_mapping.pkl"

EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

print("CHUNKS_DIR:", CHUNKS_DIR)
print("EMBEDDINGS_DIR:", EMBEDDINGS_DIR)

In [ ]:
if CHUNKS_PARQUET_PATH.exists():
    chunks_df = pd.read_parquet(CHUNKS_PARQUET_PATH)
    source_used = "parquet"
else:
    chunks_df = pd.read_csv(CHUNKS_CSV_PATH)
    source_used = "csv"

print("Loaded chunks from:", source_used)
print("chunks_df shape:", chunks_df.shape)
chunks_df.head(2)

In [ ]:
required_cols = ["chunk_id", "doc_id", "chunk_text"]

missing_cols = [c for c in required_cols if c not in chunks_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print("Unique documents:", chunks_df["doc_id"].nunique())
print("Total chunks:", len(chunks_df))

In [ ]:
EMBEDDING_MODEL = "BAAI/bge-m3"
BATCH_SIZE = 8
MAX_TEXT_PREVIEW = 120

embedding_config = {
    "embedding_model": EMBEDDING_MODEL,
    "batch_size": BATCH_SIZE,
}

embedding_config

In [ ]:
USE_PILOT_LIMIT = False
PILOT_N_CHUNKS = 100

if USE_PILOT_LIMIT:
    embed_df = chunks_df.head(PILOT_N_CHUNKS).copy().reset_index(drop=True)
else:
    embed_df = chunks_df.copy().reset_index(drop=True)

print("Chunks selected for embedding:", len(embed_df))

In [ ]:
def prepare_text_for_embedding(text: str) -> str:
    text = str(text)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

embed_df["chunk_text_clean"] = embed_df["chunk_text"].apply(prepare_text_for_embedding)
embed_df["text_len"] = embed_df["chunk_text_clean"].str.len()

embed_df[["chunk_id", "doc_id", "text_len", "chunk_text_clean"]].head(2)

In [ ]:
before_count = len(embed_df)
embed_df = embed_df[embed_df["chunk_text_clean"].str.len() > 0].copy().reset_index(drop=True)
after_count = len(embed_df)

print("Removed empty chunks:", before_count - after_count)
print("Remaining chunks:", after_count)

In [ ]:
model = SentenceTransformer(EMBEDDING_MODEL)
print("Model loaded:", EMBEDDING_MODEL)

In [ ]:
def get_embeddings_batch(texts):
    vectors = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return vectors

In [ ]:
embedding_records = []
all_vectors = []

for start_idx in tqdm(range(0, len(embed_df), BATCH_SIZE), desc="Creating embeddings"):
    batch_df = embed_df.iloc[start_idx:start_idx + BATCH_SIZE]
    batch_texts = batch_df["chunk_text_clean"].tolist()

    try:
        batch_vectors = get_embeddings_batch(batch_texts)

        for row_idx, (_, row) in enumerate(batch_df.iterrows()):
            vector = batch_vectors[row_idx]
            all_vectors.append(vector)

            embedding_records.append({
                "chunk_id": row["chunk_id"],
                "doc_id": row["doc_id"],
                "chunk_index": row.get("chunk_index"),
                "company": row.get("company"),
                "doc_type": row.get("doc_type"),
                "doc_period": row.get("doc_period"),
                "char_count": row.get("char_count"),
                "token_estimate": row.get("token_estimate"),
                "embedding_dim": len(vector),
                "text_len": row["text_len"],
                "text_preview": row["chunk_text_clean"][:MAX_TEXT_PREVIEW]
            })

    except Exception as e:
        print(f"Embedding batch failed for rows {start_idx}:{start_idx + len(batch_df)}")
        print("Reason:", e)
        raise

In [ ]:
embeddings_metadata_df = pd.DataFrame(embedding_records)
embeddings_matrix = np.array(all_vectors, dtype="float32")

print("embeddings_metadata_df shape:", embeddings_metadata_df.shape)
print("embeddings_matrix shape:", embeddings_matrix.shape)

In [ ]:
faiss.normalize_L2(embeddings_matrix)
print("Embeddings normalized.")

In [ ]:
embedding_dim = embeddings_matrix.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings_matrix)

print("FAISS index built.")
print("Index ntotal:", index.ntotal)
print("Embedding dim:", embedding_dim)

In [ ]:
row_to_chunk_id = dict(enumerate(embeddings_metadata_df["chunk_id"].tolist()))
row_to_doc_id = dict(enumerate(embeddings_metadata_df["doc_id"].tolist()))

print("Row mappings created.")
print("Sample mapping:", list(row_to_chunk_id.items())[:3])

In [ ]:
embeddings_metadata_df.to_csv(EMBEDDINGS_METADATA_CSV_PATH, index=False)
np.save(EMBEDDINGS_MATRIX_NPY_PATH, embeddings_matrix)

print("Saved metadata to:", EMBEDDINGS_METADATA_CSV_PATH)
print("Saved matrix to:", EMBEDDINGS_MATRIX_NPY_PATH)

In [ ]:
faiss.write_index(index, str(FAISS_INDEX_PATH))

with open(ROW_MAPPING_PATH, "wb") as f:
    pickle.dump({
        "row_to_chunk_id": row_to_chunk_id,
        "row_to_doc_id": row_to_doc_id
    }, f)

print("Saved FAISS index to:", FAISS_INDEX_PATH)
print("Saved row mapping to:", ROW_MAPPING_PATH)

In [ ]:
embeddings_manifest_df = pd.DataFrame([{
    "embedding_model": EMBEDDING_MODEL,
    "n_chunks_embedded": len(embeddings_metadata_df),
    "embedding_dim": int(embeddings_matrix.shape[1]),
    "faiss_index_type": "IndexFlatIP",
    "normalized_for_cosine": True,
    "metadata_csv_path": str(EMBEDDINGS_METADATA_CSV_PATH),
    "matrix_npy_path": str(EMBEDDINGS_MATRIX_NPY_PATH),
    "faiss_index_path": str(FAISS_INDEX_PATH),
    "row_mapping_path": str(ROW_MAPPING_PATH)
}])

embeddings_manifest_df

In [ ]:
embeddings_manifest_df.to_csv(EMBEDDINGS_MANIFEST_PATH, index=False)
print("Saved embeddings manifest to:", EMBEDDINGS_MANIFEST_PATH)

In [ ]:
embedding_stats = {
    "embedding_model": EMBEDDING_MODEL,
    "n_documents": int(embed_df["doc_id"].nunique()),
    "n_chunks_embedded": int(len(embed_df)),
    "embedding_dim": int(embeddings_matrix.shape[1]),
    "batch_size": BATCH_SIZE,
    "faiss_index_type": "IndexFlatIP",
    "normalized_for_cosine": True,
    "avg_text_len": float(embed_df["text_len"].mean()) if not embed_df.empty else 0
}

with open(EMBEDDINGS_STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(embedding_stats, f, ensure_ascii=False, indent=2)

print("Saved stats JSON to:", EMBEDDINGS_STATS_PATH)

In [ ]:
def embed_query(query: str):
    vector = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vector

In [ ]:
test_query = "What does 3M report about its business segments?"

query_vector = embed_query(test_query)
scores, indices = index.search(query_vector, k=5)

results = []
for score, idx in zip(scores[0], indices[0]):
    results.append({
        "score": float(score),
        "row_idx": int(idx),
        "chunk_id": row_to_chunk_id[idx],
        "doc_id": row_to_doc_id[idx]
    })

retrieval_preview_df = pd.DataFrame(results)
retrieval_preview_df


In [ ]:
preview_join_df = retrieval_preview_df.merge(
    embed_df[["chunk_id", "chunk_text"]],
    on="chunk_id",
    how="left"
)

preview_join_df

In [ ]:
for _, row in preview_join_df.iterrows():
    print("\n" + "=" * 100)
    print("chunk_id:", row["chunk_id"])
    print("score:", row["score"])
    print("-" * 100)
    print(row["chunk_text"][:1500])

In [ ]:
output_summary = pd.DataFrame([{
    "chunks_source": source_used,
    "n_documents_embedded": embed_df["doc_id"].nunique(),
    "n_chunks_embedded": len(embed_df),
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dim": int(embeddings_matrix.shape[1]),
    "metadata_csv_path": str(EMBEDDINGS_METADATA_CSV_PATH),
    "matrix_npy_path": str(EMBEDDINGS_MATRIX_NPY_PATH),
    "faiss_index_path": str(FAISS_INDEX_PATH),
    "embeddings_manifest_path": str(EMBEDDINGS_MANIFEST_PATH),
    "embeddings_stats_path": str(EMBEDDINGS_STATS_PATH)
}])

output_summary

## Συμπέρασμα

Σε αυτό το notebook:

- δημιουργήθηκαν embeddings για όλα τα chunks
- αποθηκεύτηκαν metadata και embedding matrix
- δημιουργήθηκε FAISS vector index
- εκτελέστηκε ένα πρώτο retrieval sanity check

Το επόμενο notebook θα είναι το `07_dense_retrieval.ipynb`.